# NB_lab_test_trends

Builds `adb_mediflow.gold.lab_test_trends` (one row per test type per month) from silver lab results and calendar, with a month-over-month abnormal rate trend, per the Silver to Gold STTM. Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_mediflow.gold.lab_test_trends AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                      AS year,
    month(test_date)                     AS month,
    count(*)                             AS total_tests,
    count_if(result_status = 'Abnormal') AS abnormal_test_count,
    count_if(result_status = 'Critical') AS critical_test_count
  FROM adb_mediflow.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT *,
    CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  SELECT *,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  t.test_name,
  CAST(t.year AS INT)                AS year,
  CAST(t.month AS INT)               AS month,
  c.month_name,
  c.quarter_name,
  CAST(t.total_tests AS INT)         AS total_tests,
  CAST(t.abnormal_test_count AS INT) AS abnormal_test_count,
  CAST(t.critical_test_count AS INT) AS critical_test_count,
  t.abnormal_rate_pct,
  t.critical_rate_pct,
  CAST(t.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                       AS prev_month_abnormal_rate_pct,
  CAST(t.abnormal_rate_pct - t.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag t
LEFT JOIN adb_mediflow.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(t.year, t.month, 1), 'yyyyMMdd') AS INT);

In [0]:
# %sql
# SELECT count(*) AS row_count, count(DISTINCT test_name, year, month) AS distinct_grain,
#        sum(total_tests) AS total_tests, count_if(month_name IS NULL) AS missing_calendar,
#        count_if(prev_month_abnormal_rate_pct IS NULL) AS first_month_rows, count(DISTINCT test_name) AS test_types
# FROM adb_mediflow.gold.lab_test_trends;

In [0]:
updated_df = spark.table("adb_mediflow.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))